# Project 2 — Deep Learning MCQ from Images

**Kaggle setup checklist (do this before saving/submitting):**
1. Add competition dataset (provides `images/`, `test.csv`, `sample_submission.csv`)
2. Add model: **Qwen/Qwen2-VL-7B-Instruct** from Kaggle Models Hub
   - Notebook → Add-ons → Models → search `Qwen2-VL-7B` → attach
   - Mounts at `/kaggle/input/qwen2-vl/transformers/7b-instruct/1`
3. Accelerator: **GPU (T4 x2 or P100)**
4. Internet: **OFF**

**Pipeline:**
- For each PNG image in `test.csv`: send image to Qwen2-VL-7B (offline)
- Model reads the DL MCQ printed in the image and selects option 1-4
- Outputs `submission.csv` to `/kaggle/working/`

In [ ]:
# Cell 1 — Install missing package offline
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'qwen-vl-utils', '--quiet'], check=True)
print('Dependencies ready')

In [ ]:
# Cell 2 — Imports & Paths
import re, warnings
from pathlib import Path

import pandas as pd
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

warnings.filterwarnings('ignore')

# ── Competition data ─────────────────────────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input')
COMP_DIR  = next(
    (d for d in sorted(INPUT_DIR.iterdir()) if (d/'test.csv').exists()),
    INPUT_DIR
)
IMAGES_DIR = COMP_DIR / 'images'
TEST_CSV   = COMP_DIR / 'test.csv'
SAMPLE_CSV = COMP_DIR / 'sample_submission.csv'

# ── Model (Kaggle Models Hub) ─────────────────────────────────────────────────────
MODEL_DIR = Path('/kaggle/input/qwen2-vl/transformers/7b-instruct/1')
if not MODEL_DIR.exists():
    for cfg in INPUT_DIR.rglob('config.json'):
        if 'qwen' in str(cfg).lower():
            MODEL_DIR = cfg.parent
            break

# ── Outputs ────────────────────────────────────────────────────────────────────────
SUBMIT = Path('/kaggle/working/submission.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Competition dir :', COMP_DIR)
print('Images dir      :', IMAGES_DIR)
print('Model dir       :', MODEL_DIR)
print('Device          :', DEVICE)

In [ ]:
# Cell 3 — Load Qwen2-VL-7B from local Kaggle model mount (no internet)
print(f'Loading model from {MODEL_DIR} ...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,
)
processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True
)
model.eval()
print('Model ready')

In [ ]:
# Cell 4 — Inference function
SYSTEM = (
    'You are an expert in deep learning and machine learning. '
    'You will be shown an image containing a multiple-choice question. '
    'Read the question and all four options carefully, then select the best answer.'
)
USER = (
    'This image shows a deep learning multiple-choice question with 4 options. '
    'Think step by step, then reply ONLY with the option number: 1, 2, 3, or 4. '
    'If you genuinely cannot determine the answer, reply with 5.'
)

def predict_mcq(image_path: str) -> int:
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': f'file://{image_path}'},
            {'type': 'text',  'text': USER}
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids,
                       padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=128,
            do_sample=False, temperature=None, top_p=None
        )
    raw = processor.decode(out[0][inputs.input_ids.shape[1]:],
                           skip_special_tokens=True).strip()
    print(f'    raw: {repr(raw[:80])}')
    digits = re.findall(r'\b([1-5])\b', raw)
    return int(digits[-1]) if digits else 5

In [ ]:
# Cell 5 — Run inference on all images
df   = pd.read_csv(TEST_CSV)
rows = []

for _, row in df.iterrows():
    name  = row['image_name']                              # e.g. 'image_1'
    path  = str((IMAGES_DIR / f'{name}.png').resolve())

    if not Path(path).exists():
        print(f'  {name}: NOT FOUND — skipping (option=5)')
        rows.append({'image_name': name, 'option': 5})
        continue

    print(f'  {name}:')
    pred = predict_mcq(path)
    rows.append({'image_name': name, 'option': pred})
    print(f'    → option {pred}')

sub = pd.DataFrame(rows, columns=['image_name', 'option'])
sub.to_csv(SUBMIT, index=False)
print(f'\nsubmission.csv written ({len(sub)} rows)')
print(sub)

In [ ]:
# Cell 6 — Validate submission format
result = pd.read_csv(SUBMIT)
sample = pd.read_csv(SAMPLE_CSV)
assert list(result.columns) == list(sample.columns), \
    f'Column mismatch: {result.columns.tolist()} vs {sample.columns.tolist()}'
assert len(result) == len(df), f'Row mismatch: {len(result)} vs {len(df)}'
assert result['option'].between(1,5).all(), 'Option values must be 1-5'
print('submission.csv is valid ✓')
print(result)